In [2]:
import os, json, warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')

pplt.rc.update({
    'savefig.dpi':900, 'savefig.bbox':'tight', 'savefig.pad_inches':0.02,
    'tick.minor':False, 'font.size':9, 'label.size':9, 'tick.labelsize':9,
    'title.size':9, 'abc.size':9, 'legend.fontsize':9, 'suptitle.size':9})

with open('../scripts/configs.json') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
PREDSDIR   = CONFIGS['filepaths']['predictions']
SEEDS      = CONFIGS['experiments']['nn']['seeds']
SPLIT      = 'test'

with open(os.path.join(SPLITSDIR, 'stats.json')) as f:
    STATS = json.load(f)
TP_MEAN = STATS['tp_mean']
TP_STD  = STATS['tp_std']
ZMIN    = (0.0 - TP_MEAN) / TP_STD

In [3]:
# Load test split: kernel-integrate field vars, load surface vars
splitds   = xr.open_dataset(os.path.join(SPLITSDIR, f'norm_{SPLIT}.h5'), engine='h5netcdf')
refda     = splitds['tp'].transpose('time', 'lat', 'lon')
ntime, nlat, nlon = splitds.sizes['time'], splitds.sizes.get('lat',1), splitds.sizes.get('lon',1)

fieldvars = ['rh', 'thetae', 'thetaestar']
localvars = ['lf', 'shf', 'lhf', 'sef', 'sdo', 'se']
nsig      = splitds.sizes['sig']
dsig      = splitds['dsig'].values
fields    = np.stack([splitds[v].transpose('time','lat','lon','sig').values.reshape(-1, nsig)
                      for v in fieldvars], axis=1)
surfmask  = (splitds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1, nsig)
             if 'surfmask' in splitds else None)

def kernel_integrate(fields, weights, dsig, mask=None):
    w = fields * weights[None,:,:] * dsig[None,None,:]
    if mask is not None: w *= mask[:,None,:]
    return w.sum(axis=2)

kint = np.mean([kernel_integrate(
                    fields,
                    xr.open_dataset(os.path.join(WEIGHTSDIR, f'nn_gauss_{s}_weights.nc'),
                                    engine='h5netcdf')['k'].values,
                    dsig, surfmask)
                for s in SEEDS], axis=0)   # (nsamples, 3)

FEATS = {v: kint[:, i] for i, v in enumerate(fieldvars)}
for v in localvars:
    da = splitds[v]
    FEATS[v] = (da.transpose('time','lat','lon').values.ravel() if 'time' in da.dims
                else np.tile(da.values, (ntime,1,1)).ravel())
splitds.close()

# Native-unit true precipitation
with xr.open_dataset(os.path.join(SPLITSDIR, f'{SPLIT}.h5'), engine='h5netcdf') as ds:
    TRUETP = ds.tp.load()

print('Features loaded.')

Features loaded.


In [4]:
SRFN = dict(cube=lambda x:x**3, square=lambda x:x**2, neg=lambda x:-x,
            exp=np.exp, log=np.log, abs=np.abs, sqrt=np.sqrt,
            max=np.maximum, min=np.minimum)

def eval_sr(form, extra={}):
    ns = dict(SRFN, __builtins__={}, **FEATS, **extra)
    return np.asarray(eval(form, ns), dtype=float)

def pred_to_da(arr_mm):
    """Reshape flat mm array → DataArray with refda coordinates."""
    return xr.DataArray(arr_mm.reshape(ntime, nlat, nlon),
                        dims=refda.dims, coords=refda.coords)

def sr_to_mm(expr_out):
    """Convert PySR excess-above-zmin output → native mm."""
    z = ZMIN + np.maximum(expr_out, 0)
    return np.maximum(np.expm1(z * TP_STD + TP_MEAN), 0)

def get_r2(pred_mm_da):
    ytrue, ypred = xr.align(TRUETP, pred_mm_da, join='inner')
    ss_res = ((ytrue - ypred)**2).sum(skipna=True)
    ss_tot = ((ytrue - ytrue.mean(skipna=True))**2).sum(skipna=True)
    return float(1 - ss_res / ss_tot)

# SR-MED baseline
srmed_raw = eval_sr('a * cube(max(rh, thetae - b * thetaestar - c))',
                    {'a':1.5576, 'b':1.4706, 'c':0.3756})

# NN-GAUSS predictions (already native mm)
with xr.open_dataset(os.path.join(PREDSDIR, f'nn_gauss_{SPLIT}_predictions.nc'), engine='h5netcdf') as ds:
    nn_gauss_mm = ds.tp.mean('seed').load()

print('Baselines ready.')

Baselines ready.


In [ ]:
# Candidate equations — one per run type, selected at c19-20 (maxsize limit of these runs).
# All resid/gap equations include srmed as a feature; distil is standalone.
# mul run dropped: multiplicative scale factors did not generalise (R²<0).

candidates = [
    # --- baselines ---
    dict(label='SR-MED',     color='#D42028', pred_mm=pred_to_da(sr_to_mm(srmed_raw))),
    dict(label='SR-HI',      color='#8B0000',
         pred_mm=pred_to_da(sr_to_mm(eval_sr(
             'cube(a * max(rh, thetae + b * thetaestar + c) + max(lf, shf) * d)',
             {'a':1.307, 'b':-1.3594, 'c':-0.4173, 'd':-0.1271})))),
    dict(label='NN-GAUSS',   color='#2355a1', pred_mm=nn_gauss_mm),

    # --- sr_gauss_resid c19 (seed 42: lowest loss; exp(shf) extends the c17 min pattern) ---
    dict(label='resid c19',  color='#f4a261',
         pred_mm=pred_to_da(sr_to_mm(eval_sr(
             'srmed + min((exp(shf) + -1.1003046) * (lf + -1.2116383), 1.1753592 - lhf)',
             {'srmed': srmed_raw})))),

    # --- sr_gauss_gap c20 (seed 72: lowest gap loss; srmed coupling via shf*(1+srmed)) ---
    dict(label='gap c20',    color='#e76f51',
         pred_mm=pred_to_da(sr_to_mm(eval_sr(
             '(srmed + (min(lf, 0.106686234) * (shf + (shf * srmed)))) - (lhf * 0.18459359)',
             {'srmed': srmed_raw})))),

    # --- sr_gauss_distil c20 (seeds 102 & 72 agree: SR-MED + rh correction inside cube) ---
    dict(label='distil c20', color='#457b9d',
         pred_mm=pred_to_da(sr_to_mm(eval_sr(
             'cube(max(rh, (thetaestar * -1.4840289) + (thetae + -0.2807161)) - (rh * -0.1370878)')))),
]

for c in candidates:
    c['r2'] = get_r2(c['pred_mm'])
    print(f"{c['label']:14s}  R\u00b2 = {c['r2']:.4f}")


In [ ]:
baselines = [c for c in candidates if c['label'] in ('SR-MED', 'SR-HI', 'NN-GAUSS')]
new_runs  = [c for c in candidates if c not in baselines]
ordered   = sorted(new_runs, key=lambda c: c['r2']) + baselines[::-1]

labels = [c['label'] for c in ordered]
r2s    = [c['r2']   for c in ordered]
colors = [c['color'] for c in ordered]

fig, ax = pplt.subplots(figwidth=4, figheight=2.5)
ax.barh(labels, r2s, color=colors, edgecolor='none')

r2_srmed   = next(c['r2'] for c in candidates if c['label'] == 'SR-MED')
r2_nngauss = next(c['r2'] for c in candidates if c['label'] == 'NN-GAUSS')
ax.axvline(r2_srmed,   color='#D42028', linestyle='--', linewidth=0.8, zorder=0)
ax.axvline(r2_nngauss, color='#2355a1', linestyle='--', linewidth=0.8, zorder=0)

for i, r2 in enumerate(r2s):
    ax.text(r2 + 0.003, i, f'{r2:.3f}', va='center', ha='left', fontsize=8)

ax.format(grid=False, xlabel=r'$R^2$ (native mm, test split)',
          xlim=(0, r2_nngauss + 0.12),
          title='Residual learning candidates vs baselines')
pplt.show()


In [ ]:
import pandas as pd

table_rows = [
    dict(label='SR-MED',     complexity=16, run='—',
         equation='a·cube(max(rh, thetae − b·θ★ − c))'),
    dict(label='SR-HI',      complexity=23, run='—',
         equation='cube(a·max(rh, thetae + b·θ★ + c) + max(lf,shf)·d)'),
    dict(label='NN-GAUSS',   complexity='—', run='—',
         equation='Gaussian kernel NN'),
    dict(label='resid c19',  complexity=19, run='sr_gauss_resid',
         equation='srmed + min((exp(shf)−1.10)·(lf−1.21), 1.18−lhf)'),
    dict(label='gap c20',    complexity=20, run='sr_gauss_gap',
         equation='srmed + min(lf, 0.107)·shf·(1+srmed) − 0.185·lhf'),
    dict(label='distil c20', complexity=20, run='sr_gauss_distil',
         equation='cube(max(rh, thetae − 1.484·θ★ − 0.281) + 0.137·rh)'),
]

r2_lookup = {c['label']: c['r2'] for c in candidates}
for row in table_rows:
    row['R²'] = f"{r2_lookup[row['label']]:.4f}" if row['label'] in r2_lookup else '—'

df = (pd.DataFrame(table_rows)
        .rename(columns={'label':'Label','complexity':'Complexity','run':'Run','equation':'Equation'})
        .set_index('Label'))
display(df)
